# CLOY Processing

This notebook is for you to work with the CLOY data folder.

## 1. Turning every CLOY thread into a vector of words

### A. Loading all threads

First, I want to explore looking into one thread in the CLOY file.

In [120]:
import json

with open("RedditExplorer/cloy_chronological.jsonl") as f:
    first_line = f.readline()

thread = json.loads(first_line)

print(thread.keys())
type(thread)



dict_keys(['post_id', 'timestamp', 'author', 'subreddit', 'title', 'selftext', 'url', 'score', 'permalink', 'comments'])


dict

I dump all threads into a list for simplicity later

In [121]:
threads = []

with open("RedditExplorer/cloy_chronological.jsonl") as f:
    for line in f:
        threads.append(json.loads(line))
type (threads)

list

I look into a complete thread.

In [ ]:
#threads[4]

{'post_id': 'fj5byj',
 'timestamp': 1584294837,
 'author': 'titania73',
 'subreddit': 'CrashLandingOnYou',
 'title': 'Why I think we all need a second season of CLOY. (Spoilers)',
 'selftext': 'I binged crash landing on you this week and can’t stop thinking about this show! I love the characters, the story line, and how unique it is compared to a lot of other dramedies I’ve watched. But I have to admit that I was a bit disappointed with the ending. This seems to be an unpopular opinion as a lot of fans online say that it ended well and a second season may ruin it. They found their way back to each other and that’s what matters. \n\nI disagree. I can’t believe that this is the best future we get for these two characters. Seeing each other two weeks a year may be fine for now, but how is that a fair relationship for either of them long term? So much happens in a year - how can two weeks without any other communication be enough? We see time and time again how much they mean to each other

It seems like comments are under `comment_body`, and under each comment there is also `replies`. More comments are under `comment_body` under `replies` under comments.

### B. Getting all comments

This function keeps going until it reaches a comment that does not have `replies`. This function also gets all the comments nested under one comment.

In [123]:
def get_comment_sentence (comment):
    sentence = comment["comment_body"]
    if "replies" in comment:
        for reply in comment["replies"]:
            sentence = sentence + " "+get_comment_sentence(reply)

    return sentence


### C. Turn one entire thread into one string

The question asks for main post, that means I also need to extract the `title` and `selftext`.

In [124]:
def get_thread_sentence(thread):
    sentence = thread["title"] + " " + thread["selftext"]

    for comment in thread["comments"]:
        sentence += " " + get_comment_sentence(comment)

    return sentence


Let's see if this works on the first thread.

In [ ]:
#print(get_thread_sentence(threads[1]))

ok, let’s get this started! What are your thoughts about the show? I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).

CLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. 

I’m planning on doing a rewatch soon, when I get a chance! i'm so emotionally drained and exhausted from being in awe of this show even though i finished watching 2 weeks ago. my brain doesn't work cuz i'm thinking about CLOY and my

### D. Turn the string into words using textblob

In [48]:
from textblob import TextBlob

I tokenize each thread into words.

In [55]:
thread_words = []
for thread in threads:
    text = get_thread_sentence(thread)
    tblob = TextBlob(text)
    thread_words.append(tblob.words)


In [56]:
thread_words[0]

WordList(['Crash', 'Landing', 'On', 'You', 'has', 'been', 'created', 'deleted'])

I realize that the current code makes my words case sensitive. Therefore I lowercase the texts.

In [57]:
thread_words = []
for thread in threads:
    text = get_thread_sentence(thread)
    tblob = TextBlob(text.lower())
    thread_words.append(tblob.words)

In [58]:
thread_words[0]

WordList(['crash', 'landing', 'on', 'you', 'has', 'been', 'created', 'deleted'])

### E. Create the vocabulary of all unique terms

I am a bit stuck here. I don't really know how to create the vacabulary, and I am not sure how to capture unique terms. I googled and learned about `set`

In [59]:
vocab = set()
for words in thread_words:
    for word in words:
        vocab.add(word)

In [ ]:
vocab = sorted(list(vocab))
#print(vocab[5505:7000])

['cheeky', 'cheer', 'cheered', 'cheerful', 'cheerier', 'cheering', 'cheers', 'cheers🙂', 'cheese', 'cheesey', 'cheesier', 'cheesiest', 'cheesling', 'cheesy', 'chef', 'chefs', 'cheka', 'chekhov', 'chel', 'chemical', 'chemicals', 'chemistrious', 'chemistrt', 'chemistry', 'chemistry…', 'chemistry❤️', 'chemisty', 'cheo', 'cheok-gang', 'cheol', 'cheol-gang', 'cheol-kang', 'cheol-woong', 'cheolgang', 'cheols', 'cheon', 'cheon-sang', 'cheong', 'cheool-gang', 'cherish', 'cherished', 'cherishing', 'cherry', 'chess', 'chest', 'chested', 'chests', 'cheul', 'cheul-gang', 'chew', 'chewing', 'chews', 'chez', 'chf', 'chhod', 'chi', 'chi-hwa-seon', 'chi-so', 'chi-soo', 'chi-su', 'chi-sui', 'chibi', 'chibis', 'chic', 'chicago', 'chick', 'chicken', 'chickens', 'chicks', 'chiding', 'chief', 'chihwaseon', 'child', 'child-like', 'childbirth', 'childhood', 'childhood/adolescence', 'childhood/teen', 'childish', 'childishly', 'childlike', 'children', 'childs', 'chili', 'chill', 'chillin', 'chilliness', 'chilli

### F. Create vector with the frequency

I count how many times each vocabulary word appears in each thread to create a frequency vector for each thread.

In [78]:
vectors = []
for words in thread_words:
    vector = []
    for word in vocab:
        vector.append(words.count(word))
    vectors.append(vector)
        


It has been 20 minutes and the code is still running, let's check the size of data.

In [77]:
print("number of threads:", len(thread_words))
print("number of vvocabs:", len(vocab))
print("number of words in the first thread:", len(thread_words[0]))

number of threads: 3075
number of vvocabs: 36445
number of words in the first thread: 8


It turns out that the code above ran for 100 minutes. Now let's express the frequency vector with a df.

In [79]:
import pandas as pd

df = pd.DataFrame(vectors, columns=vocab)

df.head()

,'-i,'-sshi,'d,'ll,'m,'re,'s,'ve,'🫠,0,...,🫰🫰🫰🫰,🫶,🫶🍱,🫶🏻,🫶🏻😂😂,🫶🏻🫶🏻🫶🏻,🫶🏻🫶🏻🫶🏻🫶🏻🫶🏻,🫶🏼,🫶🏽,🫶🏽✨
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,2,7,0,6,3,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,1,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,2,1,3,1,4,1,0,0,...,0,0,0,0,0,0,0,0,0,0


I am revisiting this parat after Wednesday Lab. After seeing the sample codes on the lab notebook, I think there is a more efficient approach (waiting 100 mins is terrible). Furthermore, my current vector seems to split "I'm" into `I` and `'m`, but the professor's code combines them into `im`, which makes more sense. I will retry following the lab notebook logic. 

I will keep the previous part A, B, and C.

### D2. Remove the punctuation, lowercase, and tokenize each thread

In [84]:
import string
import pandas as pd

I used logic similar to the lab notebook.

In [91]:
thread_words = []
for thread in threads:
    sentence = get_thread_sentence(thread)
    sentence_noPunct = "".join([ch for ch in sentence if not ch in string.punctuation])
    words = sentence_noPunct.lower().split()
    thread_words.append(words)



### E2. Create vocabulary for unique terms

I used vocab.append() and got an error, I learned about `vocab.update()`.

In [96]:
vocab = set()
for words in thread_words:
    vocab.update(words)

vocab = sorted(vocab)

print("Number of threads:", len(thread_words))
print("Vocabulary size:", len(vocab))
print(vocab[:20])

Number of threads: 3075
Vocabulary size: 39333
['0', '00000001', '00001', '0001', '0010', '0011', '0030', '0038', '007', '01', '014', '0140', '017', '01823', '01httpswwwgentlemonstercomshopver1detailphpitid1572394578', '01x16', '02', '020', '0200', '02021988']


### F2. Create a frequency vector for every thread

In [97]:
vectors = []
for words in thread_words:
    vector = []
    for word in vocab:
        vector.append(words.count(word))
    vectors.append(vector)


Yay! It only took less than 5 minutes this time.

### G2. Display Frequency vector on a df

In [98]:
df = pd.DataFrame(vectors, columns=vocab)
df.head()

,0,00000001,00001,0001,0010,0011,0030,0038,007,01,...,🫰🫰🫰🫰,🫶,🫶🍱,🫶🏻,🫶🏻😂😂,🫶🏻🫶🏻🫶🏻,🫶🏻🫶🏻🫶🏻🫶🏻🫶🏻,🫶🏼,🫶🏽,🫶🏽✨
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 2. Calculate the similarity score between every two threads, and save the dataframe into a CSV file.

### A. Define cosine similarity function

I copied the code from the lecture notes. I originally used a nested for loops and it was taking forever. This function is much more efficient.

In [105]:
import numpy as np
from numpy.linalg import norm

def cosineSimilarity(vec1, vec2):
    """Calculate the cosine similarity between two vectors."""
    V1 = np.array(vec1)
    V2 = np.array(vec2)
    cosine = np.dot(V1, V2) / (norm(V1) * norm(V2))
    return cosine

### B. Calculate the Cosine Similarity

I will run a for loop to call the cosine similarity formula.

In [ ]:
similarities = []

for i in range(len(vectors)):
    row = []

    for j in range(len(vectors)):
        similarity = cosineSimilarity(vectors[i], vectors[j])
        row.append(similarity)

    similarities.append(row)

The code above ran for 600 minutes and is still not done yet. I need to look for another more efficient approach. I googled about `scikit-learn`. It is much more efficient and only took 2 steps.

In [114]:
from sklearn.metrics.pairwise import cosine_similarity

In [115]:
similarities = cosine_similarity(vectors)
similarities_df = pd.DataFrame(similarities)
similarities_df.head(20)


,0,1,2,3,4,5,6,7,8,9,...,3065,3066,3067,3068,3069,3070,3071,3072,3073,3074
0,1.000000,0.057374,0.030890,0.010004,0.061422,0.024456,0.017851,0.068425,0.092685,0.039528,...,0.060059,0.073213,0.125000,0.135565,0.125868,0.017157,0.059827,0.015988,0.083853,0.044544
1,0.057374,1.000000,0.525775,0.806039,0.795403,0.637483,0.796317,0.871319,0.635587,0.449042,...,0.821379,0.496221,0.043030,0.537850,0.690618,0.797749,0.809850,0.584933,0.408394,0.205302
2,0.030890,0.525775,1.000000,0.496912,0.453334,0.362612,0.412472,0.503718,0.373013,0.195366,...,0.549151,0.445077,0.061780,0.435512,0.491454,0.481918,0.551798,0.414858,0.193403,0.154107
3,0.010004,0.806039,0.496912,1.000000,0.678696,0.542157,0.702914,0.770414,0.590241,0.411261,...,0.746915,0.430081,0.040016,0.464558,0.605417,0.724527,0.756828,0.546377,0.355677,0.203200
4,0.061422,0.795403,0.453334,0.678696,1.000000,0.555183,0.652910,0.806582,0.590757,0.440261,...,0.815206,0.525238,0.053232,0.570451,0.621783,0.749548,0.792723,0.476086,0.489858,0.253896
5,0.024456,0.637483,0.362612,0.542157,0.555183,1.000000,0.543095,0.629899,0.453336,0.286143,...,0.551503,0.386746,0.048912,0.436421,0.487589,0.538179,0.552023,0.347213,0.328109,0.104577
6,0.017851,0.796317,0.412472,0.702914,0.652910,0.543095,1.000000,0.757322,0.586186,0.341530,...,0.676478,0.406728,0.044629,0.439126,0.568923,0.691353,0.731755,0.504611,0.317341,0.178117
7,0.068425,0.871319,0.503718,0.770414,0.806582,0.629899,0.757322,1.000000,0.620272,0.407704,...,0.812961,0.513830,0.046817,0.564907,0.667246,0.785593,0.817625,0.534331,0.421966,0.218165
8,0.092685,0.635587,0.373013,0.590241,0.590757,0.453336,0.586186,0.620272,1.000000,0.213541,...,0.564127,0.330370,0.013241,0.366827,0.458643,0.560946,0.604899,0.365809,0.325678,0.141549
9,0.039528,0.449042,0.195366,0.411261,0.440261,0.286143,0.341530,0.407704,0.213541,1.000000,...,0.525595,0.296347,0.079057,0.358544,0.378129,0.383394,0.420491,0.369082,0.274004,0.154945


It only took 1.4s to calculate all the cosine similarities.

### C. Save the df into a csv file

This is a straightforward step!

In [117]:
similarities_df.to_csv("CosSimilarity.csv",index = True)

I could not push the csv file to github, because it is larger than 100 MB.

## 3. Given one thread ID, gives top 5 most similar threads

In [ ]:
print(threads[1]["post_id"])
#print(threads[1])


ffh52z
{'post_id': 'ffh52z', 'timestamp': 1583693959, 'author': 'bruisedlee123', 'subreddit': 'CrashLandingOnYou', 'title': 'ok, let’s get this started! What are your thoughts about the show?', 'selftext': 'I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-com and drama genre down pat! I also loved Capt. Ri’s squad, most especially Pyo Chi Su ☺️ (we can probably talk about individual characters next time).\n\nCLOY tackled politics, drama and deception, family issues, friendships - and best of all, pure unconditional love, love brought about by fate - all in one highly entertaining and well-written package. \n\nI’m planning on doing a rewatch soon, when I get a chance!', 'url': 'https://www.reddit.

### A. Search for the index of the thread given its ID

I will use the `threads[1]` as an example for this question.

In [225]:
def get_id_thread(id):
   for i in range(len(threads)):
      if threads[i]["post_id"] == id:
         return i


id_index = get_id_thread("ffh52z")

### B. Find the top 5 most similar threads

Gives indices from smallest to largest, then reverse those resulting indices.

In [226]:
print(id_index)
np.argsort(similarities[id_index])[::-1]

1


array([   1,  452,  574, ..., 2855, 2872, 1776], shape=(3075,))

We want to exclude the first one, since it is the thread itself. The thread's cosine similarity with itself is 1.


In [227]:
five_similar = np.argsort(similarities[id_index])[::-1][1:6]
five_similar

array([ 452,  574, 2201, 2332,  387])

### C. Print the 5 most similar threads' ID and subject title

In [228]:
#threads[i]
i = 0
for ind in five_similar :
    i=1+i
    print("Top " +str(i)+ " most similar thread's ID: "+threads[ind]['post_id'] +" ; Subject title: " + threads[ind]['title'] )
   

Top 1 most similar thread's ID: i42ez5 ; Subject title: How come you started watching it?
Top 2 most similar thread's ID: iy5qeu ; Subject title: Withdrawal, anyone?
Top 3 most similar thread's ID: 17tpx8l ; Subject title: I just finished “Crash Landing On You” and I will never be the same.
Top 4 most similar thread's ID: 1ci4cmh ; Subject title: I have just finished binge-watching Crash Landing On You. I am overwhelmed.
Top 5 most similar thread's ID: hp9elr ; Subject title: post-series depression


### D. Combine A, B, and C into one function


In [230]:
def find_similarities (id):

    # A. find i
    for i in range(len(threads)):
        if threads[i]["post_id"] == id:
            id_ind = i
            print(id_ind)

    # B. find five_similar
    five_similar = np.argsort(similarities[id_ind])[::-1][1:6]
    print(id_ind)
    print(five_similar)

    # C. print them
    j = 0
    for ind in five_similar :
        j+=1
        print("Top " +str(j)+ " most similar thread's ID: "+threads[ind]['post_id'] +" ; Subject title: " + threads[ind]['title'] )

In [231]:
find_similarities("ffh52z")


1
1
[ 452  574 2201 2332  387]
Top 1 most similar thread's ID: i42ez5 ; Subject title: How come you started watching it?
Top 2 most similar thread's ID: iy5qeu ; Subject title: Withdrawal, anyone?
Top 3 most similar thread's ID: 17tpx8l ; Subject title: I just finished “Crash Landing On You” and I will never be the same.
Top 4 most similar thread's ID: 1ci4cmh ; Subject title: I have just finished binge-watching Crash Landing On You. I am overwhelmed.
Top 5 most similar thread's ID: hp9elr ; Subject title: post-series depression


Yay! The answers are consistent with section A, B, and C.

## 4. Verification

##### I used the thread explorer to examine the five most similar threads to `ffh52z`. The original thread is mainly a general discussion of viewers’ experiences with CLOY, including their love for the characters and story, emotional reactions, and overall appreciation of the show. The five most similar threads seemed reasonable because they discussed many of the same themes, such as discovering CLOY, becoming emotionally attached to the show, loving the characters, rewatching it, and struggling to move on after finishing it. Although each thread focused on a slightly different topic, they reflected many of the main themes discussed in the original thread.

#### However, since all of the threads come from the same CLOY subreddit, I am not fully convinced by only looking at the five most similar threads. Many threads may naturally use similar words and discuss similar topics because they are all about the same show. To further check how well the cosine similarity works, I also want to look at the **least similar thread to `ffh52z`** and compare its content with the original thread. If the least similar thread discusses noticeably different topics, this would give me more confidence that the similarity measure is actually distinguishing between different discussions.

In [234]:
least_similar = np.argsort(similarities[id_index])[0]
least_similar
print(threads[least_similar]["post_id"])
print(threads[least_similar]["title"])

ttn4es
Login • Instagram


#### This thread is deleted, so let me look for more.

In [236]:
least_similars = np.argsort(similarities[id_index])[0:5]
least_similars
j = 0
for ind in least_similars :
    j+=1
    print("Top " +str(j)+ " least similar thread's ID: "+threads[ind]['post_id'] +" ; Subject title: " + threads[ind]['title'] )


Top 1 least similar thread's ID: ttn4es ; Subject title: Login • Instagram
Top 2 least similar thread's ID: 1nilhlv ; Subject title: Speechless
Top 3 least similar thread's ID: 1n4jfwo ; Subject title: RJH 💔 YSR
Top 4 least similar thread's ID: h7g9hv ; Subject title: CCG’s military position?
Top 5 least similar thread's ID: kz14r2 ; Subject title: Fanfics!!


#### They are all deleted, so let me look for even more threads.

In [245]:
least_similarss = np.argsort(similarities[id_index])[30:40]
least_similarss
j = 0
for ind in least_similarss :
    j+=1
    print("Top " +str(j)+ " least similar thread's ID: "+threads[ind]['post_id'] +" ; Subject title: " + threads[ind]['title'] )

Top 1 least similar thread's ID: 1hyrxyi ; Subject title: Son Ye-jin's Birthday Instagram Post
Top 2 least similar thread's ID: mi0htz ; Subject title: It’s finally happening??
Top 3 least similar thread's ID: k2gyk0 ; Subject title: Lady...[Spoilers, Ep 13]
Top 4 least similar thread's ID: jlroby ; Subject title: Happy Halloween!!!! Uwu
Top 5 least similar thread's ID: 1qx3zat ; Subject title: BTS from sets
Top 6 least similar thread's ID: jbwzwf ; Subject title: Man Bok
Top 7 least similar thread's ID: 1ko78xu ; Subject title: Absolutely stunning frames
Top 8 least similar thread's ID: jpode3 ; Subject title: Dangerous Landings Great Pilots
Top 9 least similar thread's ID: 1lfucj3 ; Subject title: Park Su-Chan
Top 10 least similar thread's ID: 19dlwh0 ; Subject title: Pyo Chi-su?!?!


#### I initially compared `ffh52z` with its least similar threads and found that they were mostly deleted or contained very few words. This result makes sense because threads with very little text have fewer words that can overlap with ffh52z, resulting in low cosine similarity. This gives me some confidence that the cosine similarity calculation is behaving as expected. 

#### However, I also want to test whether it can distinguish between threads that contain substantial amounts of text. Therefore, I will examine some threads that rank lower than the top five but still contain enough content, and compare their topics with ffh52z.

In [248]:
len(similarities)

less_similars = np.argsort(similarities[id_index])[::-1][400:406]
j = 0
for ind in less_similars :
    j+=1
    print("Thread's ID: "+threads[ind]['post_id'] +" ; Subject title: " + threads[ind]['title'] )

Thread's ID: 1ri1aqc ; Subject title: 2026 CLOY (re)watch Ep. 3 Discussion
Thread's ID: lfmyh6 ; Subject title: DVD / Blu-Ray Boxset?
Thread's ID: koy4kt ; Subject title: OST - Flower made me teary!
Thread's ID: 1rcamj7 ; Subject title: Character Question: The soldiers and the Tin Man, the Scarecrow, and the Cowardly Lion
Thread's ID: 1n093pc ; Subject title: Favorite CLOY Episode
Thread's ID: lfdhrf ; Subject title: Biggest Tearjerker?


#### To further test the similarity results, I looked at several threads ranked around 400th most similar to ffh52z. Unlike the top five, which shared many of the original thread’s main themes, these threads tended to focus on one narrower topic, such as a specific episode, the OST, favorite emotional scenes, or buying a DVD/Blu-ray. They still had some overlap with ffh52z, which makes sense because they are all discussions about CLOY, but the overlap was much more limited. This comparison gives me more confidence that the cosine similarity is meaningfully distinguishing between discussions within the same subreddit, rather than simply identifying that they are all about the same show.

## 5. Investigate Stop Words

### A. Remove the stop words

I googled for the stop word list and the library it is at. I defined the stop words.

In [254]:
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))



[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/lejunli/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


I want to first define a function that removes stop words.

In [256]:
def remove_stopwords(words):
    # create an empty list
    wordlist= []
    for word in words:
        if word.lower() not in stop_words:
            wordlist.append(word)
    return wordlist

I want to remove stopwords for all threads

In [257]:
filtered_thread_words = []

for words in thread_words:
    removed_stopwords = remove_stopwords(words)
    filtered_thread_words.append(removed_stopwords)

Let's do a sanity check!'

In [260]:
print(len(thread_words[1]))
print(len(filtered_thread_words[1]))
print(thread_words[1][:50])
print(filtered_thread_words[1][:50])

1535
794
['ok', 'let’s', 'get', 'this', 'started', 'what', 'are', 'your', 'thoughts', 'about', 'the', 'show', 'i', 'haven’t', 'seen', 'a', 'kdrama', 'for', 'the', 'longest', 'time', 'and', 'it', 'was', 'actually', 'my', 'dad', 'who', 'chose', 'to', 'stream', 'cloy', 'on', 'netflix', 'he', 'stopped', 'watching', 'after', 'the', 'second', 'episode', 'but', 'i', 'went', 'on', 'after', 'immediately', 'falling', 'in', 'love']
['ok', 'let’s', 'get', 'started', 'thoughts', 'show', 'haven’t', 'seen', 'kdrama', 'longest', 'time', 'actually', 'dad', 'chose', 'stream', 'cloy', 'netflix', 'stopped', 'watching', 'second', 'episode', 'went', 'immediately', 'falling', 'love', 'leads', 'hyun', 'bin', 'charismatic', 'capt', 'ri', 'jeong', 'hyeok', 'son', 'yejin', 'perfect', 'yoon', 'seri', 'she’s', 'definitely', 'got', 'romcom', 'drama', 'genre', 'pat', 'also', 'loved', 'capt', 'ri’s', 'squad']


In [268]:
filtered_vocab = set()
for words in filtered_thread_words:
    for word in words:
        filtered_vocab.add(word)

filtered_vocab = sorted(list(filtered_vocab))

Now-- Create the vector, same as the previous part!

Update: I tried using the code from F2 section, but it is inefficient as it takes more than 4 minutes to load. I looked up for a faster way using Counter.

In [279]:
from collections import Counter

filtered_vectors = []

for words in filtered_thread_words:
    counts = Counter(words)
    filtered_vector = []

    for word in filtered_vocab:
        filtered_vector.append(counts[word])

    filtered_vectors.append(filtered_vector)
    


Finally, display the vectors on a df

In [280]:
df = pd.DataFrame(filtered_vectors, columns=filtered_vocab)
df.head()

,0,00000001,00001,0001,0010,0011,0030,0038,007,01,...,🫰🫰🫰🫰,🫶,🫶🍱,🫶🏻,🫶🏻😂😂,🫶🏻🫶🏻🫶🏻,🫶🏻🫶🏻🫶🏻🫶🏻🫶🏻,🫶🏼,🫶🏽,🫶🏽✨
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### B. Recalculate the cosine similarity

In [281]:
filtered_similarities = cosine_similarity(filtered_vectors)
filtered_similarities_df = pd.DataFrame(filtered_similarities)
filtered_similarities_df.head(20)

,0,1,2,3,4,5,6,7,8,9,...,3065,3066,3067,3068,3069,3070,3071,3072,3073,3074
0,1.000000,0.000000,0.000000,0.000000,0.026162,0.000000,0.000000,0.013558,0.000000,0.000000,...,0.000000,0.000000,0.188982,0.063246,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,1.000000,0.122788,0.356546,0.423038,0.119753,0.397293,0.472193,0.127897,0.124776,...,0.436439,0.185925,0.039841,0.204000,0.216600,0.366505,0.370293,0.213444,0.125988,0.090142
2,0.000000,0.122788,1.000000,0.186653,0.076189,0.035806,0.030039,0.146654,0.036980,0.048280,...,0.191391,0.104530,0.052414,0.114018,0.122663,0.068730,0.139073,0.033968,0.000000,0.040893
3,0.000000,0.356546,0.186653,1.000000,0.229654,0.052885,0.190144,0.287222,0.089732,0.132433,...,0.336071,0.097045,0.000000,0.111035,0.127418,0.240956,0.285058,0.105120,0.090929,0.094912
4,0.026162,0.423038,0.076189,0.229654,1.000000,0.067551,0.165761,0.369610,0.158718,0.122965,...,0.449203,0.143960,0.000000,0.153882,0.117488,0.383012,0.392810,0.128168,0.215762,0.123437
5,0.000000,0.119753,0.035806,0.052885,0.067551,1.000000,0.006991,0.157532,0.017213,0.022473,...,0.067948,0.068119,0.048795,0.073485,0.061489,0.063984,0.086930,0.021082,0.015430,0.038069
6,0.000000,0.397293,0.030039,0.190144,0.165761,0.006991,1.000000,0.295155,0.140798,0.018854,...,0.215349,0.106130,0.000000,0.102748,0.121593,0.196134,0.197838,0.145912,0.045308,0.047907
7,0.013558,0.472193,0.146654,0.287222,0.369610,0.157532,0.295155,1.000000,0.159082,0.070805,...,0.431648,0.179869,0.051245,0.216088,0.199263,0.374235,0.368286,0.172695,0.100472,0.107948
8,0.000000,0.127897,0.036980,0.089732,0.158718,0.017213,0.140798,0.159082,1.000000,0.034816,...,0.150490,0.045227,0.000000,0.050596,0.027217,0.078791,0.106018,0.059876,0.063746,0.019659
9,0.000000,0.124776,0.048280,0.132433,0.122965,0.022473,0.018854,0.070805,0.034816,1.000000,...,0.140486,0.065608,0.065795,0.088077,0.130289,0.092913,0.114722,0.028427,0.083225,0.102665


### C. Use the same thread `ffh52z` and find its new top 5


In [289]:
def find_filtered_similarities (id):

    # A. find i
    for i in range(len(threads)):
        if threads[i]["post_id"] == id:
            id_ind = i
            print(id_ind)

    # B. find five_filtered_similar
    five_filtered_similar = np.argsort(filtered_similarities[id_ind])[::-1][1:6]

    # C. print them
    j = 0
    for ind in five_filtered_similar :
        j+=1
        print("Top " +str(j)+ " most similar thread's ID: "+threads[ind]['post_id'] +" ; Subject title: " + threads[ind]['title'] )

In [290]:
find_filtered_similarities("ffh52z")

1
Top 1 most similar thread's ID: 17tpx8l ; Subject title: I just finished “Crash Landing On You” and I will never be the same.
Top 2 most similar thread's ID: i42ez5 ; Subject title: How come you started watching it?
Top 3 most similar thread's ID: kclv6n ; Subject title: Happy Birthday to CLOY!
Top 4 most similar thread's ID: iy5qeu ; Subject title: Withdrawal, anyone?
Top 5 most similar thread's ID: u1lawc ; Subject title: Just finished the show…


### D.Conclusion -- Comparing with the unfiltered function

Response from earlier section:  
Top 1 most similar thread's ID: i42ez5 ; Subject title: How come you started watching it?  
Top 2 most similar thread's ID: iy5qeu ; Subject title: Withdrawal, anyone?  
Top 3 most similar thread's ID: 17tpx8l ; Subject title: I just finished “Crash Landing On You” and I will never be the same.  
Top 4 most similar thread's ID: 1ci4cmh ; Subject title: I have just finished binge-watching Crash Landing On You. I am overwhelmed.  
Top 5 most similar thread's ID: hp9elr ; Subject title: post-series depression

Newest response from filtered similarity:  
Top 1 most similar thread's ID: 17tpx8l ; Subject title: I just finished “Crash Landing On You” and I will never be the same.  
Top 2 most similar thread's ID: i42ez5 ; Subject title: How come you started watching it?  
Top 3 most similar thread's ID: kclv6n ; Subject title: Happy Birthday to CLOY!  
Top 4 most similar thread's ID: iy5qeu ; Subject title: Withdrawal, anyone?  
Top 5 most similar thread's ID: u1lawc ; Subject title: Just finished the show…

#### After removing stop words, the top five most similar threads changed, but three of the original top five remained. Considering that there are 3,075 threads in the dataset, this suggests that the similarity results are **fairly consistent** even after stop words are removed. However, the order changed and two new threads appeared in the top five, showing that stop words still have some influence on the cosine similarity scores. Overall, removing stop words seems to preserve the main similarity patterns while allowing the comparison to focus more on meaningful content words.